# 衝突率とスループット解析ノート

このノートブックは、PIM と Slotted Aloha のログを読み込み、
衝突率と **1ノードあたりスループット** を比較するためのものです。

## 1. ライブラリの読み込み

解析とプロットに必要なライブラリを読み込みます。

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import japanize_matplotlib
import pandas as pd
import yaml

## 2. PDFの出力先設定

ログの入力元と、図を保存するPDFファイルの出力先をここで指定します。
必要に応じてこのセルだけ変更してください。

In [ ]:
# ログパス
PIM_LOG_DIR = Path("tsv/pim_20260427_003109")
SLOTTED_ALOHA_LOG_DIR = Path("logs/slotted_aloha_20260427_003335")

# PDF出力先
OUTPUT_DIR = Path("img/analysis")
COLLISION_RATE_PDF = OUTPUT_DIR / "collision_rate_comparison_nodes_1_30.pdf"
THROUGHPUT_PER_NODE_BPS_PDF = OUTPUT_DIR / "throughput_per_node_bps_comparison_nodes_1_30.pdf"

## 3. 解析コード

イベントログを読み込んで、ノード数ごとの衝突率と **1ノードあたりスループット** を計算します。

In [ ]:
def load_config(config_path: Path) -> dict:
    """設定ファイルを読み込み、解析に必要なパラメータ辞書を返す。"""
    with config_path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def load_events(events_path: Path) -> pd.DataFrame:
    """events_N.tsvを読み込み、時刻とラウンドを数値列へ整形して返す。"""
    df = pd.read_csv(events_path, sep="\t")
    df["time_tu"] = pd.to_numeric(df["time_tu"], errors="coerce")
    df["round"] = pd.to_numeric(df["round"], errors="coerce").astype(int)
    df["event"] = df["event"].astype(str)
    df["result"] = df["result"].astype(str)
    return df


def iter_event_files(log_dir: Path) -> list[tuple[int, Path]]:
    """log_dir配下の events_N.tsv を (N, Path) で返す。"""
    files = []
    for path in log_dir.glob("events_*.tsv"):
        m = re.fullmatch(r"events_(\d+)\.tsv", path.name)
        if m is None:
            continue
        files.append((int(m.group(1)), path))
    return sorted(files, key=lambda x: x[0])


def success_event_mask(protocol: str, events: pd.DataFrame) -> pd.Series:
    """プロトコルごとの成功イベント条件を返す。"""
    if protocol == "pim":
        return events["event"] == "rx_success"
    if protocol == "slotted_aloha":
        return (events["event"] == "data_response") & (events["result"] == "success")
    raise ValueError(f"未知のプロトコルです: {protocol}")


def parse_detail_fields(detail: str) -> dict:
    """detail文字列を key=value の辞書へ変換する。"""
    out = {}
    for item in str(detail).split(","):
        part = item.strip()
        if "=" not in part:
            continue
        key, value = part.split("=", 1)
        out[key.strip()] = value.strip()
    return out


def extract_slotted_aloha_first_inventory(block: pd.DataFrame) -> pd.DataFrame:
    """
    Slotted Aloha ブロックから、mask_len=0 の最初の inventory だけを切り出す。

    返す範囲は最初の inventory_cmd から、次の inventory_cmd の直前まで。
    inventory_cmd が1回しかない場合は、その inventory 全体を返す。
    """
    inventory_cmds = block[(block["phase"] == "inventory") & (block["event"] == "inventory_cmd")]
    first_cmd = inventory_cmds[inventory_cmds["detail"].astype(str).str.contains("mask_len=0", na=False)]

    if first_cmd.empty:
        return block.iloc[0:0]

    start_time = float(first_cmd.iloc[0]["time_tu"])
    next_cmds = inventory_cmds[inventory_cmds["time_tu"] > start_time]
    if next_cmds.empty:
        end_time = None
    else:
        end_time = float(next_cmds["time_tu"].min())

    inventory = block[(block["phase"] == "inventory") & (block["time_tu"] >= start_time)]
    if end_time is not None:
        inventory = inventory[inventory["time_tu"] < end_time]

    return inventory


def summarize_protocol(protocol: str, log_dir: Path) -> pd.DataFrame:
    """
    1プロトコル分のログから、nodes別の衝突率と1ノードあたりスループット(bps)を算出する。

    衝突率: collision_packets / (collision_packets + successes)
    1ノードあたりスループット(bps): total_throughput_bps / nodes
    """
    config = load_config(log_dir / "config.yaml")
    data_bits = float(config["data_bits"])
    tu_sec = float(config["tu"])
    event_files = iter_event_files(log_dir)

    if not event_files:
        raise FileNotFoundError(f"events_N.tsv が見つかりません: {log_dir}")

    rows = []
    for num_nodes, events_path in event_files:
        events = load_events(events_path)
        if events.empty:
            continue

        success_mask = success_event_mask(protocol, events)
        successes = int(success_mask.sum())
        if protocol == "slotted_aloha":
            collision_block = extract_slotted_aloha_first_inventory(events)
            collision_successes = int(
                ((collision_block["event"] == "tag_response") & (collision_block["result"] == "success")).sum()
            )
        else:
            collision_block = events
            collision_successes = successes

        collisions = int((collision_block["event"] == "collision").sum())
        collision_rows = collision_block[collision_block["event"] == "collision"]
        collision_packets = 0
        for detail in collision_rows["detail"]:
            fields = parse_detail_fields(detail)
            try:
                collision_packets += int(float(fields.get("count", "1")))
            except ValueError:
                collision_packets += 1

        attempts = collision_successes + collision_packets
        collision_rate = (collision_packets / attempts) if attempts > 0 else 0.0

        round_last_tu = events.groupby("round")["time_tu"].max()
        num_rounds = int(round_last_tu.shape[0])
        avg_round_tu = float(round_last_tu.mean()) if num_rounds > 0 else 0.0
        avg_round_sec = avg_round_tu * tu_sec
        avg_successes_per_round = (successes / num_rounds) if num_rounds > 0 else 0.0

        total_throughput_bps = (
            (avg_successes_per_round * data_bits / avg_round_sec)
            if avg_round_sec > 0
            else 0.0
        )
        throughput_per_node_bps = (total_throughput_bps / num_nodes) if num_nodes > 0 else 0.0

        row = {
            "protocol": protocol,
            "nodes": int(num_nodes),
            "collisions": collisions,
            "successes": successes,
            "collision_rate": collision_rate,
            "throughput_per_node_bps": throughput_per_node_bps,
        }
        rows.append(row)

    return pd.DataFrame(rows).sort_values(["nodes", "protocol"]).reset_index(drop=True)


def plot_collision_rate(df: pd.DataFrame, output_pdf: Path) -> None:
    """nodesに対する衝突率をPIM/Slotted Alohaで比較し、PDFに出力する。"""
    pivot = df.pivot(index="nodes", columns="protocol", values="collision_rate").sort_index()

    fig, ax = plt.subplots(figsize=(8, 5))
    for protocol in pivot.columns:
        ax.plot(pivot.index, pivot[protocol], marker="o", label=protocol)

    ax.set_title("Collision Rate vs Selected Tags")
    ax.set_xlabel("Selected Tags (nodes)")
    ax.set_ylabel("Collision Rate")
    ax.set_ylim(0.0, 1.0)
    ax.grid(alpha=0.3)
    ax.legend()

    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    with PdfPages(output_pdf) as pdf:
        pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def plot_throughput_per_node_bps(df: pd.DataFrame, output_pdf: Path) -> None:
    """nodesに対する1ノードあたりスループット(bps)を比較し、PDFに出力する。"""
    pivot = df.pivot(index="nodes", columns="protocol", values="throughput_per_node_bps").sort_index()

    fig, ax = plt.subplots(figsize=(8, 5))
    for protocol in pivot.columns:
        ax.plot(pivot.index, pivot[protocol], marker="o", label=protocol)

    ax.set_title("Throughput Per Node (bps) vs Selected Tags")
    ax.set_xlabel("Selected Tags (nodes)")
    ax.set_ylabel("Throughput Per Node (bps)")
    ax.grid(alpha=0.3)
    ax.legend()

    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    with PdfPages(output_pdf) as pdf:
        pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

## 4. 実行

このセルを実行すると、比較解析を行ってPDFを出力し、結果テーブルを表示します。

In [ ]:
pim_df = summarize_protocol("pim", PIM_LOG_DIR)
slotted_df = summarize_protocol("slotted_aloha", SLOTTED_ALOHA_LOG_DIR)
merged = pd.concat([pim_df, slotted_df], ignore_index=True)

plot_collision_rate(merged, COLLISION_RATE_PDF)
plot_throughput_per_node_bps(merged, THROUGHPUT_PER_NODE_BPS_PDF)

print("比較解析が完了しました。")
print(f"PIM log dir: {PIM_LOG_DIR}")
print(f"Slotted Aloha log dir: {SLOTTED_ALOHA_LOG_DIR}")
print(f"Collision rate PDF: {COLLISION_RATE_PDF}")
print(f"Throughput per node PDF: {THROUGHPUT_PER_NODE_BPS_PDF}")
print()
print("=== 解析結果 ===")
display(merged[["protocol", "nodes", "collision_rate", "throughput_per_node_bps"]])